# 03 - Final modeling and predictions

Train the selected LightGBM model on the clean Jan-Oct development data, predict the 12,000 validation loads, and generate the December chart inputs.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config
from src import evaluation as ev
from src import features as F
from src import preprocessing as pp
from src.train_predict import train_final_model, write_outputs

pd.set_option("display.max_columns", 30)

## Modeling contract

- Target: `log(posted_rate)`
- Model: LightGBM quantile-style L1 regression
- Features: distance, log distance, weight, missing-weight flag, equipment, day of week, and route coordinates
- Training data: 47,323 clean Jan-Oct rows
- Outputs: positive predictions for validation and December

In [ ]:
train = pd.read_csv(config.TRAIN_CLEAN, parse_dates=["date"])
holdout_train = train[train.date < config.HOLDOUT_START].copy()
holdout_test = train[train.date >= config.HOLDOUT_START].copy()

from lightgbm import LGBMRegressor
model = LGBMRegressor(**config.LGBM_FINAL_PARAMS)
model.fit(
    F.build_features(holdout_train, config.FINAL_FEATURE_GROUPS),
    np.log(holdout_train[config.TARGET]),
)
holdout_prediction = np.exp(model.predict(F.build_features(holdout_test, config.FINAL_FEATURE_GROUPS)))
ev.metrics(holdout_test[config.TARGET], holdout_prediction)

## Final fit

The holdout is used only as an internal reference. The final model is now refit on all clean Jan-Oct rows.

In [ ]:
model, validation, december_raw, X_validation, X_december = train_final_model()
predictions_path = ROOT / "validation_predictions.csv"
december_path = ROOT / "december_predictions.csv"
write_outputs(
    model, validation, december_raw, X_validation, X_december,
    predictions_path, december_path,
)
print(predictions_path)
print(december_path)

In [ ]:
predictions = pd.read_csv(predictions_path)
december = pd.read_csv(december_path)
print(predictions.shape, predictions.predicted_rate.describe().round(2).to_dict())
print(december.shape, december.predicted_rate.describe().round(2).to_dict())
assert (predictions.predicted_rate > 0).all()
assert (december.predicted_rate > 0).all()

## Scorer command

Run from the repository root:

```powershell
.\.venv\Scripts\python.exe scorer\score.py --predictions validation_predictions.csv --december-predictions december_predictions.csv
```